# GEO Semantic Search — Colab trainer ($0)

Fine-tunes **two** sentence-embedding models — `all-MiniLM-L6-v2` and `bge-small-en-v1.5` — for semantic SEO/GEO retrieval with `MultipleNegativesRankingLoss`, then reports IR metrics (Recall@k, MRR@10, nDCG@10) **base vs fine-tuned** on a fixed seed=42 held-out split.

No QLoRA / no bitsandbytes — the models are small enough for a full fine-tune, so the stack stays minimal.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

### 1. Clone the repo

In [ ]:
import os
if not os.path.exists('geo-semantic-search'):
    !git clone https://github.com/ippongod/geo-semantic-search.git
%cd geo-semantic-search
!git pull -q

### 2. Install dependencies (minimal — no bitsandbytes)
If the pinned install can't be satisfied on this Colab image, it falls back to the latest compatible `sentence-transformers` 3.x (same Trainer API and same `cosine_*` metric keys).

In [ ]:
!pip install -q -r requirements.txt || pip install -q "sentence-transformers>=3.3,<4" datasets "accelerate>=0.26"

### 3. Dependency guard + GPU check
`sentence-transformers` 3.x needs `transformers < 5`. Recent Colab images may ship 5.x; this cell fails **loudly** (not silently) and tells you to restart if a fix was applied.

In [ ]:
import subprocess, sys
from packaging.version import Version
import transformers
if Version(transformers.__version__) >= Version('5'):
    print(f'transformers {transformers.__version__} >= 5 — installing a compatible build...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers>=4.41,<5'], check=True)
    raise SystemExit('Installed transformers<5. Now Runtime > Restart session, then Run all again.')
import sentence_transformers
print('transformers       :', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
!nvidia-smi -L

### 4. Train — all-MiniLM-L6-v2

In [ ]:
!python scripts/train.py --base sentence-transformers/all-MiniLM-L6-v2 --out models/geo-minilm

### 5. Train — bge-small-en-v1.5

In [ ]:
!python scripts/train.py --base BAAI/bge-small-en-v1.5 --out models/geo-bge

### 6. Evaluate base vs fine-tuned (IR metrics)
Both models use the **same** seed=42 held-out split, so the comparison is fair.

In [ ]:
!python scripts/eval.py --base sentence-transformers/all-MiniLM-L6-v2 --tuned models/geo-minilm --tag minilm
!python scripts/eval.py --base BAAI/bge-small-en-v1.5 --tuned models/geo-bge --tag bge

### 7. Fill results into the README & show the report

In [ ]:
!python scripts/fill_results.py
print(open('results.md', encoding='utf-8').read())

### 8. Quick semantic-search sanity check (fine-tuned MiniLM)

In [ ]:
!python scripts/infer.py --model models/geo-minilm --query "where can I get single-origin coffee beans delivered"

### 9. Download the two fine-tuned models
Unzip locally into `models/` to run `scripts/infer.py`. (Optionally commit `results.json` / `results.md` back to the repo.)

In [ ]:
import shutil
from google.colab import files
for name in ['geo-minilm', 'geo-bge']:
    shutil.make_archive(name, 'zip', f'models/{name}')
    files.download(f'{name}.zip')